In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / 'output_v2'
FIGS_DIR = PROJECT_ROOT / 'notebook' / 'figs'
FIGS_DIR.mkdir(parents=True, exist_ok=True)


# Backward-compatible alias used by legacy plotting cells
NOTEBOOK_DIR = FIGS_DIR



In [2]:
def load_data():
    print('Loading V2 data...')

    users = pd.read_csv(OUTPUT_DIR / 'users.csv')
    groups = pd.read_csv(OUTPUT_DIR / 'groups.csv', keep_default_na=False)
    memberships = pd.read_csv(OUTPUT_DIR / 'group_memberships.csv', keep_default_na=False)
    shared_expenses = pd.read_csv(OUTPUT_DIR / 'shared_expenses.csv', keep_default_na=False)
    splits = pd.read_csv(OUTPUT_DIR / 'expense_splits.csv', keep_default_na=False)

    settlements_path = OUTPUT_DIR / 'settlements.csv'
    if settlements_path.exists():
        settlements = pd.read_csv(settlements_path, keep_default_na=False)
    else:
        settlements = pd.read_csv(OUTPUT_DIR / 'recurring_payments.csv', keep_default_na=False)

    expenses = pd.read_csv(OUTPUT_DIR / 'recurring_payments.csv', keep_default_na=False)

    # ---- Normalize users schema to legacy EDA names ----
    user_rename = {
        'financial_persona': 'persona',
        'monthly_income_npr': 'income',
        'city_tier': 'location_tier',
        'spend_multiplier': 'spending_multiplier',
    }
    users = users.rename(columns=user_rename)
    if 'join_date' not in users.columns:
        users['join_date'] = pd.Timestamp('2026-01-01')

    # ---- Normalize expenses schema to legacy EDA names ----
    if 'expected_amount_npr' in expenses.columns and 'amount' not in expenses.columns:
        expenses['amount'] = pd.to_numeric(expenses['expected_amount_npr'], errors='coerce').fillna(0.0)
    if 'is_recurring' not in expenses.columns:
        expenses['is_recurring'] = True
    if 'is_group_expense' not in expenses.columns:
        expenses['is_group_expense'] = False
    if 'category' not in expenses.columns:
        expenses['category'] = 'General'

    # Synthetic daily date if none exists in v2 recurring table
    if 'date' not in expenses.columns:
        base = pd.Timestamp('2026-01-01')
        expenses['date'] = [base + pd.Timedelta(days=i) for i in range(len(expenses))]

    expenses['date'] = pd.to_datetime(expenses['date'], errors='coerce')
    expenses['amount'] = pd.to_numeric(expenses['amount'], errors='coerce').fillna(0.0)
    expenses['is_group_expense'] = expenses['is_group_expense'].astype(bool)
    expenses['is_recurring'] = expenses['is_recurring'].astype(bool)
    expenses['year'] = expenses['date'].dt.year
    expenses['month'] = expenses['date'].dt.month
    expenses['day_of_week'] = expenses['date'].dt.dayofweek
    expenses['day_of_month'] = expenses['date'].dt.day
    expenses['is_weekend'] = expenses['day_of_week'].isin([5, 6]).astype(int)

    users['join_date'] = pd.to_datetime(users['join_date'], errors='coerce')

    print(f'  Users: {len(users):,}')
    print(f'  Expenses (V2 mapped): {len(expenses):,}')
    print(f'  Groups: {len(groups):,}')
    print(f'  Shared expenses: {len(shared_expenses):,}')
    print(f'  Splits: {len(splits):,}')

    return users, groups, memberships, expenses, splits, settlements



In [3]:
def analyze_user_distributions(users):
    print("\n" + "=" * 60)
    print("USER DISTRIBUTIONS")
    print("=" * 60)

    persona_col = 'persona' if 'persona' in users.columns else ('financial_persona' if 'financial_persona' in users.columns else None)
    income_col = 'income' if 'income' in users.columns else ('monthly_income_npr' if 'monthly_income_npr' in users.columns else None)
    location_col = 'location_tier' if 'location_tier' in users.columns else ('city_tier' if 'city_tier' in users.columns else None)
    spend_mult_col = 'spending_multiplier' if 'spending_multiplier' in users.columns else ('spend_multiplier' if 'spend_multiplier' in users.columns else None)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    if persona_col:
        users[persona_col].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
        axes[0].set_title('Persona Distribution')
        axes[0].tick_params(axis='x', rotation=45)

    if income_col:
        users[income_col].hist(bins=50, ax=axes[1], color='coral')
        axes[1].set_title('Income Distribution')
        axes[1].set_xlabel('Income')

    if 'age' in users.columns:
        users['age'].hist(bins=30, ax=axes[2], color='seagreen')
        axes[2].set_title('Age Distribution')

    if location_col:
        users[location_col].value_counts().plot(kind='bar', ax=axes[3], color='mediumpurple')
        axes[3].set_title('Location Tier')

    if spend_mult_col:
        users[spend_mult_col].hist(bins=50, ax=axes[4], color='goldenrod')
        axes[4].set_title('Spending Multiplier Distribution')

    if 'join_date' in users.columns:
        users['join_date'].hist(bins=50, ax=axes[5], color='teal')
        axes[5].set_title('User Join Date Distribution')

    plt.tight_layout()
    plt.savefig(NOTEBOOK_DIR / "01_user_distributions.png", bbox_inches='tight')
    plt.close()

    if income_col:
        print(f"  Income: mean={users[income_col].mean():,.0f}, median={users[income_col].median():,.0f}")
    if 'age' in users.columns:
        print(f"  Age: mean={users['age'].mean():.1f}, median={users['age'].median():.0f}")
    if spend_mult_col:
        print(f"  Spending multiplier: mean={users[spend_mult_col].mean():.2f}, std={users[spend_mult_col].std():.2f}")



In [4]:
def analyze_expense_patterns(expenses, users):
    print("\n" + "=" * 60)
    print("EXPENSE PATTERNS")
    print("=" * 60)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    cat_counts = expenses['category'].value_counts()
    cat_counts.plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('Expense Count by Category')

    cat_totals = expenses.groupby('category')['amount'].sum().sort_values(ascending=False)
    cat_totals.plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_title('Total Spend by Category')

    expenses['amount'].clip(upper=500).hist(bins=100, ax=axes[2], color='seagreen')
    axes[2].set_title('Amount Distribution (capped at $500)')
    axes[2].set_xlabel('Amount ($)')

    monthly = expenses.groupby(expenses['date'].dt.to_period('M'))['amount'].agg(['sum', 'count'])
    monthly['sum'].plot(ax=axes[3], color='mediumpurple')
    axes[3].set_title('Monthly Total Spend')
    axes[3].set_xlabel('Month')

    monthly['count'].plot(ax=axes[4], color='goldenrod')
    axes[4].set_title('Monthly Transaction Count')
    axes[4].set_xlabel('Month')

    dow_spend = expenses.groupby('day_of_week')['amount'].mean()
    dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    dow_spend.index = dow_labels
    dow_spend.plot(kind='bar', ax=axes[5], color='teal')
    axes[5].set_title('Average Spend by Day of Week')

    plt.tight_layout()
    plt.savefig(NOTEBOOK_DIR / "02_expense_patterns.png", bbox_inches='tight')
    plt.close()

    print(f"  Total expenses: {len(expenses):,}")
    print(f"  Total amount: ${expenses['amount'].sum():,.0f}")
    print(f"  Mean expense: ${expenses['amount'].mean():.2f}")
    print(f"  Median expense: ${expenses['amount'].median():.2f}")
    print(f"  Individual: {(~expenses['is_group_expense']).sum():,}, Group: {expenses['is_group_expense'].sum():,}")
    print(f"  Negative amounts: {(expenses['amount'] < 0).sum():,}")


In [5]:
def analyze_temporal_trends(expenses):
    print("\n" + "=" * 60)
    print("TEMPORAL TRENDS")
    print("=" * 60)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    yearly = expenses.groupby('year').agg(
        total_spend=('amount', 'sum'),
        avg_spend=('amount', 'mean'),
        count=('amount', 'count'),
    )
    yearly['total_spend'].plot(ax=axes[0], marker='o', color='steelblue')
    axes[0].set_title('Yearly Total Spend (Inflation Effect)')
    axes[0].set_ylabel('Total ($)')

    yearly['avg_spend'].plot(ax=axes[1], marker='o', color='coral')
    axes[1].set_title('Yearly Average Expense')
    axes[1].set_ylabel('Avg ($)')

    monthly_cat = expenses.groupby(['year', 'month', 'category'])['amount'].sum().reset_index()
    monthly_cat['yearmonth'] = monthly_cat['year'].astype(str) + '-' + monthly_cat['month'].astype(str).str.zfill(2)
    top_cats = expenses['category'].value_counts().head(5).index
    for cat in top_cats:
        cat_data = monthly_cat[monthly_cat['category'] == cat].sort_values('yearmonth')
        cat_data.plot(x='yearmonth', y='amount', ax=axes[2], label=cat)
    axes[2].set_title('Top 5 Categories - Monthly Spend Trend')
    axes[2].get_xaxis().set_ticks([])
    axes[2].set_ylabel('Monthly Spend ($)')

    dom_spend = expenses.groupby('day_of_month')['amount'].mean()
    dom_spend.plot(ax=axes[3], color='teal')
    axes[3].set_title('Average Spend by Day of Month (Payday Effect)')
    axes[3].set_ylabel('Avg ($)')
    axes[3].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Post-payday')
    axes[3].legend()

    plt.tight_layout()
    plt.savefig(NOTEBOOK_DIR / "03_temporal_trends.png", bbox_inches='tight')
    plt.close()

    print(f"  Year-over-year growth in avg spend:")
    for yr in sorted(expenses['year'].unique()):
        avg = expenses[expenses['year'] == yr]['amount'].mean()
        print(f"    {yr}: ${avg:.2f}")


In [6]:
def analyze_persona_spending(expenses, users):
    print("\n" + "=" * 60)
    print("PERSONA-LEVEL SPENDING")
    print("=" * 60)

    persona_col = 'persona' if 'persona' in users.columns else ('financial_persona' if 'financial_persona' in users.columns else None)
    income_col = 'income' if 'income' in users.columns else ('monthly_income_npr' if 'monthly_income_npr' in users.columns else None)
    location_col = 'location_tier' if 'location_tier' in users.columns else ('city_tier' if 'city_tier' in users.columns else None)

    merge_cols = ['user_id']
    if persona_col: merge_cols.append(persona_col)
    if income_col: merge_cols.append(income_col)
    if location_col: merge_cols.append(location_col)
    if 'age' in users.columns: merge_cols.append('age')

    merged = expenses.merge(users[merge_cols], on='user_id', how='left')

    if persona_col and persona_col != 'persona':
        merged = merged.rename(columns={persona_col: 'persona'})

    if 'persona' not in merged.columns:
        print('Persona column not found, skipping persona-level charts.')
        return

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    persona_spend = merged.groupby('persona').agg(
        avg_per_txn=('amount', 'mean'),
        total=('amount', 'sum'),
        count=('amount', 'count'),
        std=('amount', 'std'),
    ).sort_values('avg_per_txn', ascending=True)

    persona_spend['avg_per_txn'].plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('Avg Expense by Persona')

    for i, persona in enumerate(persona_spend.index):
        p_data = merged[merged['persona'] == persona]
        daily = p_data.groupby('date')['amount'].sum()
        axes[1].hist(daily.clip(upper=500), bins=50, alpha=0.5, label=persona)
    axes[1].set_title('Daily Spending Distribution by Persona')
    axes[1].legend(fontsize=8)
    axes[1].set_xlabel('Daily Spend')

    if 'category' in merged.columns:
        persona_cat = merged.groupby(['persona', 'category'])['amount'].sum().groupby(level=0).apply(
            lambda x: 100 * x / x.sum()
        ).unstack(fill_value=0)
        persona_cat.plot(kind='bar', stacked=True, ax=axes[2], colormap='tab20')
        axes[2].set_title('Category Mix by Persona (%)')
        axes[2].legend(fontsize=6, bbox_to_anchor=(1.05, 1))
        axes[2].tick_params(axis='x', rotation=45)

    if location_col and location_col in merged.columns:
        loc_spend = merged.groupby(location_col)['amount'].mean()
        loc_spend.plot(kind='bar', ax=axes[3], color=['coral', 'seagreen', 'steelblue'][:len(loc_spend)])
        axes[3].set_title('Avg Spend by Location Tier')

    plt.tight_layout()
    plt.savefig(NOTEBOOK_DIR / "04_persona_spending.png", bbox_inches='tight')
    plt.close()



In [7]:
def analyze_group_dynamics(expenses, groups, memberships):
    print("\n" + "=" * 60)
    print("GROUP DYNAMICS")
    print("=" * 60)

    group_expenses = expenses[expenses['is_group_expense']].copy()

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    group_type_counts = groups['group_type'].value_counts()
    group_type_counts.plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Group Type Distribution')
    axes[0].tick_params(axis='x', rotation=45)

    if len(group_expenses) > 0:
        group_cat = group_expenses.merge(groups[['group_id', 'group_type']], on='group_id', how='left')
        type_spend = group_cat.groupby('group_type')['amount'].mean().sort_values(ascending=True)
        type_spend.plot(kind='barh', ax=axes[1], color='coral')
        axes[1].set_title('Avg Group Expense by Type')

        type_count = group_cat.groupby('group_type')['amount'].count()
        type_count.plot(kind='bar', ax=axes[2], color='seagreen')
        axes[2].set_title('Group Transaction Count by Type')
        axes[2].tick_params(axis='x', rotation=45)

    members_per_group = memberships.groupby('group_id')['user_id'].count()
    members_per_group.hist(bins=30, ax=axes[3], color='mediumpurple')
    axes[3].set_title('Members Per Group Distribution')
    axes[3].set_xlabel('Number of Members')

    plt.tight_layout()
    plt.savefig(NOTEBOOK_DIR / "05_group_dynamics.png", bbox_inches='tight')
    plt.close()

    print(f"  Group expenses: {len(group_expenses):,} ({len(group_expenses)/len(expenses)*100:.1f}%)")
    print(f"  Individual expenses: {(~expenses['is_group_expense']).sum():,}")
    print(f"  Avg group size: {members_per_group.mean():.1f}")
    print(f"  Group size range: {members_per_group.min()} to {members_per_group.max()}")


In [8]:
def analyze_data_quality(expenses):
    print("\n" + "=" * 60)
    print("DATA QUALITY (for cleaning)")
    print("=" * 60)

    print(f"  Rows: {len(expenses):,}")
    print(f"  Columns: {len(expenses.columns)}")

    # Missingness summary
    missing = expenses.isna().mean().sort_values(ascending=False)
    top_missing = missing[missing > 0].head(10)
    if len(top_missing) > 0:
        print("  Top missing columns (%):")
        for c, v in top_missing.items():
            print(f"    {c}: {v*100:.2f}%")
    else:
        print("  No missing values detected.")

    # Outlier check on amount if present
    if 'amount' in expenses.columns:
        amt = expenses['amount'].astype(float)
        q1, q3 = amt.quantile(0.25), amt.quantile(0.75)
        iqr = q3 - q1
        if iqr > 0:
            outliers = ((amt < (q1 - 1.5*iqr)) | (amt > (q3 + 1.5*iqr))).sum()
            print(f"  Amount outliers (IQR rule): {outliers:,}")

    # Description stats (schema-safe)
    if 'description' in expenses.columns:
        desc = expenses['description'].fillna('').astype(str)
        empty_desc = (desc.str.strip() == '').sum()
        print(f"  Empty descriptions: {empty_desc:,}")
    else:
        print("  Description column not present in this V2-mapped dataset.")



In [9]:
def analyze_spending_hour(expenses):
    print("\n" + "=" * 60)
    print("SPENDING HOUR ANALYSIS")
    print("=" * 60)

    # V1 used created_at; V2-mapped data may only have date
    if 'created_at' in expenses.columns:
        ts = pd.to_datetime(expenses['created_at'], errors='coerce')
    elif 'date' in expenses.columns:
        ts = pd.to_datetime(expenses['date'], errors='coerce')
    else:
        print('No timestamp column (created_at/date) available; skipping hour analysis.')
        return

    # If only dates exist, hourly view is not meaningful
    if ts.dt.hour.nunique() <= 1:
        print('Timestamp has no meaningful hour granularity in this dataset; skipping hourly chart.')
        return

    hour_df = expenses.copy()
    hour_df['hour'] = ts.dt.hour
    hour_spend = hour_df.groupby('hour')['amount'].agg(['count', 'mean', 'sum'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    hour_spend['count'].plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Expense Count by Hour')
    axes[0].set_xlabel('Hour')

    hour_spend['mean'].plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Average Amount by Hour')
    axes[1].set_xlabel('Hour')

    plt.tight_layout()
    plt.savefig(NOTEBOOK_DIR / '06_hourly_patterns.png', bbox_inches='tight')
    plt.close()



In [10]:
def main():
    users, groups, memberships, expenses, splits, settlements = load_data()

    analyze_user_distributions(users)
    analyze_expense_patterns(expenses, users)
    analyze_temporal_trends(expenses)
    analyze_persona_spending(expenses, users)
    analyze_group_dynamics(expenses, groups, memberships)
    analyze_data_quality(expenses)
    analyze_spending_hour(expenses)

    print("\n" + "=" * 60)
    print("EDA COMPLETE - Figures saved to notebook/figs/")
    print("=" * 60)


if __name__ == "__main__":
    try:
        main()
    except FileNotFoundError as e:
        print(f"Missing input file: {e}")
        print("Run dataset generation first so output_v2/*.csv exists.")
        print("Suggested order: 01_dataset_creation_v2.ipynb -> 01_eda.ipynb")
    except Exception as e:
        print(f"Runtime error: {e}")



Loading V2 data...
  Users: 300
  Expenses (V2 mapped): 39,755
  Groups: 120
  Shared expenses: 25,237
  Splits: 121,468

USER DISTRIBUTIONS
  Income: mean=86,565, median=70,450
  Age: mean=40.4, median=39
  Spending multiplier: mean=1.00, std=0.27

EXPENSE PATTERNS
  Total expenses: 39,755
  Total amount: $409,340,879
  Mean expense: $10296.59
  Median expense: $4720.99
  Individual: 39,755, Group: 0
  Negative amounts: 0

TEMPORAL TRENDS
  Year-over-year growth in avg spend:
    2026: $19099.78
    2027: $4288.87
    2028: $6681.23
    2029: $17680.05
    2030: $5620.18
    2031: $7118.62
    2032: $19168.54
    2033: $6969.28
    2034: $7096.21
    2035: $15272.88
    2036: $9621.73
    2037: $6608.52
    2038: $14297.80
    2039: $10939.31
    2040: $7345.64
    2041: $14105.21
    2042: $11330.21
    2043: $5675.67
    2044: $13533.31
    2045: $10456.64
    2046: $6615.26
    2047: $13987.84
    2048: $9731.56
    2049: $7009.71
    2050: $12742.73
    2051: $11461.41
    2052: $